# Credit Card Fraud Risk-Level Prediction

This notebook provides a complete analysis of the credit card fraud dataset and trains a machine learning model to predict fraud risk. The final prediction output includes fraud probability as a percentage and a risk level: **Low**, **Medium**, or **High**.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve
)
import joblib

plt.rcParams['figure.figsize'] = (8, 5)
sns.set_theme(style='whitegrid')

## 2. Load Dataset

In [ ]:
DATA_PATH = Path('../data/credit_card_fraud_10k.csv')

# This fallback helps the notebook run even when opened from the project root.
if not DATA_PATH.exists():
    DATA_PATH = Path('data/credit_card_fraud_10k.csv')

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Dataset Overview

In [ ]:
print('Dataset shape:', df.shape)
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.isnull().sum()

## 4. Target Variable Analysis

The target variable is `is_fraud`, where 1 means fraudulent and 0 means legitimate.

In [ ]:
target_counts = df['is_fraud'].value_counts().rename(index={0: 'Legitimate', 1: 'Fraud'})
target_percent = (df['is_fraud'].value_counts(normalize=True) * 100).round(2).rename(index={0: 'Legitimate', 1: 'Fraud'})

target_summary = pd.DataFrame({
    'count': target_counts,
    'percentage': target_percent
})
target_summary

In [ ]:
sns.countplot(data=df, x='is_fraud')
plt.title('Distribution of Fraud vs Legitimate Transactions')
plt.xlabel('Is Fraud')
plt.ylabel('Count')
plt.show()

## 5. Exploratory Data Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop(['transaction_id', 'is_fraud'], errors='ignore')

for col in numeric_cols:
    sns.histplot(data=df, x=col, hue='is_fraud', kde=True, bins=30)
    plt.title(f'{col} Distribution by Fraud Status')
    plt.show()

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in categorical_cols:
    fraud_rate = df.groupby(col)['is_fraud'].mean().sort_values(ascending=False) * 100
    fraud_rate.plot(kind='bar')
    plt.title(f'Fraud Rate by {col}')
    plt.ylabel('Fraud Rate (%)')
    plt.xlabel(col)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
corr = df.drop(columns=['transaction_id'], errors='ignore').select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## 6. Prepare Features and Target

In [ ]:
X = df.drop(columns=['is_fraud', 'transaction_id'], errors='ignore')
y = df['is_fraud']

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

## 7. Build and Train Machine Learning Pipeline

A Random Forest model is used with class balancing because fraud detection datasets are commonly imbalanced.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    min_samples_leaf=2,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)

## 8. Model Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 4))
print('Average Precision:', round(average_precision_score(y_test, y_prob), 4))
print('
Classification Report:')
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, y_prob):.4f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob)
plt.plot(recall, precision, label=f'Avg Precision = {average_precision_score(y_test, y_prob):.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

## 9. Fraud Probability as Percentage and Risk Level

The model generates a fraud probability between 0 and 1. This notebook converts that probability into a percentage and assigns a risk category.

Risk-level rule:

- **Low risk:** fraud probability below 20%
- **Medium risk:** fraud probability from 20% to below 60%
- **High risk:** fraud probability 60% and above

In [ ]:
def assign_risk_level(fraud_probability_percent):
    if fraud_probability_percent < 20:
        return 'Low'
    elif fraud_probability_percent < 60:
        return 'Medium'
    else:
        return 'High'

results = X_test.copy()
results['actual_is_fraud'] = y_test.values
results['fraud_probability_percent'] = (y_prob * 100).round(2)
results['legitimate_probability_percent'] = ((1 - y_prob) * 100).round(2)
results['fraud_risk_level'] = results['fraud_probability_percent'].apply(assign_risk_level)

results.sort_values('fraud_probability_percent', ascending=False).head(20)

In [ ]:
risk_distribution = results['fraud_risk_level'].value_counts().reindex(['Low', 'Medium', 'High']).fillna(0).astype(int)
risk_distribution

In [ ]:
risk_distribution.plot(kind='bar')
plt.title('Predicted Fraud Risk-Level Distribution')
plt.xlabel('Risk Level')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=0)
plt.show()

## 10. Save Model and Prediction Output

In [ ]:
Path('../models').mkdir(exist_ok=True)
Path('../reports').mkdir(exist_ok=True)

joblib.dump(pipeline, '../models/fraud_risk_model.pkl')
results.to_csv('../reports/fraud_risk_predictions.csv', index=False)

print('Saved model to ../models/fraud_risk_model.pkl')
print('Saved predictions to ../reports/fraud_risk_predictions.csv')

## 11. Sample Business Interpretation

Transactions with a high fraud probability should be prioritized for manual review or additional verification. Medium-risk transactions may require monitoring or secondary checks, while low-risk transactions can usually continue through the normal approval process. This approach helps convert a technical model output into an operational fraud-risk decision.